# 커스텀 미들웨어 예제: 도구 실행 감사 리포트 (책에 없는 예제)

책 6장에서는 `before_model`/`after_model` 로깅, `wrap_model_call` 재시도·동적 모델 선택 예제를 다뤘습니다.
이 노트북은 그중 다루지 않았던 **`wrap_tool_call`을 노드 스타일 후크(`before_agent`, `after_agent`)와 한 클래스 안에서 조합**하는 예제입니다.

## 만들 것: `ToolAuditMiddleware`

에이전트가 실행하는 모든 도구 호출을 감사(audit)해서, 이름·인자·소요 시간·성공 여부를 기록하고
실행이 끝나면 요약 리포트를 출력하는 클래스형 커스텀 미들웨어입니다.

| 후크 | 역할 |
|---|---|
| `before_agent` | 이번 실행의 감사 로그를 초기화 |
| `wrap_tool_call` | 각 도구 호출을 감싸서 시간 측정 + 성공/실패 기록. **실패해도 예외를 삼키고 `ToolMessage(status="error")`로 변환**해서 에이전트가 죽지 않게 함 |
| `after_agent` | 수집된 로그로 요약 리포트 출력 |

`wrap_tool_call`이 도구에서 발생한 예외(`ZeroDivisionError` 등)를 실제로 전달받는지, 그리고 그걸 흡수해서
에이전트를 안 죽게 만들 수 있는지까지 직접 검증하며 만든 예제입니다.

**모델**: 로컬 Ollama `llama3.1:8b` (API 키 불필요)

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

import time

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.chat_models import init_chat_model
from langchain.messages import ToolMessage
from langchain.tools import tool

## 1. 도구 정의

`get_weather`는 항상 성공하고, `risky_divide`는 `b=0`이면 `ZeroDivisionError`를 던집니다 — 실패 케이스를 일부러 만든 것입니다.

In [2]:
@tool
def get_weather(city: str) -> str:
    """도시의 날씨를 조회합니다."""
    time.sleep(0.3)  # 실제 API 호출을 흉내
    return f"{city}는 맑음, 22도"


@tool
def risky_divide(a: float, b: float) -> float:
    """두 수를 나눕니다."""
    return a / b  # b가 0이면 예외 발생

## 2. `ToolAuditMiddleware` 정의

`self.call_log`는 인스턴스 상태입니다 — `before_agent`에서 초기화하고, `wrap_tool_call`이 호출될 때마다 쌓이고,
`after_agent`에서 읽습니다. 여러 후크가 상태를 공유해야 하는 상황이라 클래스 기반이 아니면 만들기 까다로운 예제입니다.

In [3]:
class ToolAuditMiddleware(AgentMiddleware):
    """도구 호출을 감사해서 실행 후 요약 리포트를 남기는 커스텀 미들웨어."""

    def __init__(self):
        super().__init__()
        self.call_log = []

    def before_agent(self, state, runtime):
        self.call_log = []  # 이번 실행 로그 초기화
        print("🔍 감사 시작")
        return None

    def wrap_tool_call(self, request, handler):
        tool_name = request.tool.name if request.tool else request.tool_call["name"]
        args = request.tool_call["args"]
        start = time.time()
        try:
            response = handler(request)
            self.call_log.append(
                {"tool": tool_name, "args": args, "duration": time.time() - start, "status": "success"}
            )
            return response
        except Exception as e:
            self.call_log.append(
                {
                    "tool": tool_name,
                    "args": args,
                    "duration": time.time() - start,
                    "status": "error",
                    "error": str(e),
                }
            )
            # 예외를 그대로 올리지 않고 ToolMessage로 흡수 -> 에이전트가 죽지 않고 계속 진행
            return ToolMessage(
                content=f"도구 실행 실패: {e}",
                tool_call_id=request.tool_call["id"],
                name=tool_name,
                status="error",
            )

    def after_agent(self, state, runtime):
        total = len(self.call_log)
        success = sum(1 for c in self.call_log if c["status"] == "success")
        total_time = sum(c["duration"] for c in self.call_log)
        print(f"\n📋 감사 리포트 — 총 {total}건 (성공 {success} / 실패 {total - success}), 총 소요 {total_time:.2f}초")
        for c in self.call_log:
            mark = "✅" if c["status"] == "success" else "❌"
            extra = f" — {c.get('error')}" if c["status"] == "error" else ""
            print(f"  {mark} {c['tool']}({c['args']}) — {c['duration']:.2f}s{extra}")
        return None

## 3. 에이전트 생성 및 실행

날씨 조회(성공)와 0으로 나누기(실패)를 한 번에 요청해서, 성공/실패가 섞인 감사 리포트를 만들어 봅니다.
이 예제는 interrupt가 없는 순수 미들웨어 데모라 체크포인터가 필요 없습니다.

In [4]:
audit = ToolAuditMiddleware()

model = init_chat_model("ollama:llama3.1:8b", temperature=0)
# Claude API를 쓰려면 위 줄 대신:
# model = init_chat_model("claude-sonnet-4-5")

agent = create_agent(
    model=model,
    tools=[get_weather, risky_divide],
    middleware=[audit],
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Check the weather in Seoul, then use risky_divide to compute "
                    "10 divided by 0. Try both tools."
                ),
            }
        ]
    }
)

print("\n최종 응답:")
print(result["messages"][-1].content)

🔍 감사 시작

📋 감사 리포트 — 총 2건 (성공 1 / 실패 1), 총 소요 0.31초
  ❌ risky_divide({'a': '10', 'b': '0'}) — 0.00s — float division by zero
  ✅ get_weather({'city': 'Seoul'}) — 0.31s

최종 응답:
The weather in Seoul is clear with a temperature of 22 degrees Celsius.

However, attempting to divide 10 by 0 using the `risky_divide` tool resulted in an error because division by zero is undefined.


## 관찰 포인트

- `risky_divide`가 실패했지만 **에이전트 전체가 죽지 않고** 최종 응답까지 정상적으로 나왔습니다 — `wrap_tool_call`이 예외를
  흡수해서 `ToolMessage(status="error")`로 바꿔줬기 때문입니다.
- 모델은 이 에러 `ToolMessage`를 읽고, 최종 답변에서 "0으로 나눌 수 없다"는 사실을 스스로 설명합니다 — 개발자가 별도로
  에러 처리 로직을 프롬프트에 넣지 않아도, 미들웨어가 만든 에러 메시지를 모델이 그대로 활용합니다.
- `after_agent`의 리포트는 도구 호출 순서와 무관하게(병렬 실행이든 순차 실행이든) **모든 호출을 빠짐없이** 기록합니다 —
  `self.call_log`가 인스턴스 상태로 유지되기 때문입니다.